# DECISION TREE (ID3) WITH SCRATCH

In [1]:
#ID3

import pandas as pd
import numpy as np

data = pd.read_csv("iris.csv")

df = pd.DataFrame(data)
#print(df)
df.drop(columns = 'Id', inplace = True)

train_data = df.sample(frac = 0.8, random_state = 1)
test_data = df.drop(train_data.index)

#print(train_data)
#print(test_data)

#df.info()

X_train = train_data[['SepalLengthCm',  'SepalWidthCm',  'PetalLengthCm',  'PetalWidthCm']]
Y_train = train_data[['Species']]

X_test = test_data[['SepalLengthCm',  'SepalWidthCm',  'PetalLengthCm',  'PetalWidthCm']]
Y_test = test_data[['Species']]

print(df)

X = df[['SepalLengthCm',  'SepalWidthCm',  'PetalLengthCm',  'PetalWidthCm']]
Y = df[['Species']]


     SepalLengthCm  SepalWidthCm  PetalLengthCm  PetalWidthCm         Species
0              5.1           3.5            1.4           0.2     Iris-setosa
1              4.9           3.0            1.4           0.2     Iris-setosa
2              4.7           3.2            1.3           0.2     Iris-setosa
3              4.6           3.1            1.5           0.2     Iris-setosa
4              5.0           3.6            1.4           0.2     Iris-setosa
..             ...           ...            ...           ...             ...
145            6.7           3.0            5.2           2.3  Iris-virginica
146            6.3           2.5            5.0           1.9  Iris-virginica
147            6.5           3.0            5.2           2.0  Iris-virginica
148            6.2           3.4            5.4           2.3  Iris-virginica
149            5.9           3.0            5.1           1.8  Iris-virginica

[150 rows x 5 columns]


In [2]:
# Discretize continuous features
for column in df.columns[:-1]:
    df[column] = pd.cut(df[column], bins=3, labels=['low', 'medium', 'high'])  # 3 bins as an example

# Check the discretized data
print(df)

np.unique(Y['Species'])

df['SepalLengthCm'] = df['SepalLengthCm'].map({'low': 1, 'medium': 2, 'high': 3})
df['SepalWidthCm'] = df['SepalWidthCm'].map({'low': 1, 'medium': 2, 'high': 3})
df['PetalLengthCm'] = df['PetalLengthCm'].map({'low': 1, 'medium': 2, 'high': 3})
df['PetalWidthCm'] = df['PetalWidthCm'].map({'low': 1, 'medium': 2, 'high': 3})

df['Species'] = df['Species'].map({'Iris-setosa': 1, 'Iris-virginica': 2, 'Iris-versiculor': 3})

# Check the results
#print(X)
#print(Y)
print()






    SepalLengthCm SepalWidthCm PetalLengthCm PetalWidthCm         Species
0             low       medium           low          low     Iris-setosa
1             low       medium           low          low     Iris-setosa
2             low       medium           low          low     Iris-setosa
3             low       medium           low          low     Iris-setosa
4             low       medium           low          low     Iris-setosa
..            ...          ...           ...          ...             ...
145        medium       medium          high         high  Iris-virginica
146        medium          low          high         high  Iris-virginica
147        medium       medium          high         high  Iris-virginica
148        medium       medium          high         high  Iris-virginica
149        medium       medium          high         high  Iris-virginica

[150 rows x 5 columns]



In [3]:
#ID3

import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

def entropy_func(c, n):
    """
    The math formula
    """
    return -(c*1.0/n)*math.log(c*1.0/n, 2)

def entropy_cal(c1, c2):
    """
    Returns entropy of a group of data
    c1: count of one class
    c2: count of another class
    """
    if c1== 0 or c2 == 0:  # when there is only one class in the group, entropy is 0
        return 0
    return entropy_func(c1, c1+c2) + entropy_func(c2, c1+c2)

# get the entropy of one big circle showing above
def entropy_of_one_division(division): 
    """
    Returns entropy of a divided group of data
    Data may have multiple classes
    """
    s = 0
    n = len(division)
    classes = set(division)
    for c in classes:   # for each class, get entropy
        n_c = sum(division==c)
        e = n_c*1.0/n * entropy_cal(sum(division==c), sum(division!=c)) # weighted avg
        s += e
    return s, n

# The whole entropy of two big circles combined
def get_entropy(y_predict, y_real):
    """
    Returns entropy of a split
    y_predict is the split decision, True/Fasle, and y_true can be multi class
    """
    if len(y_predict) != len(y_real):
        print('They have to be the same length')
        return None
    n = len(y_real)
    s_true, n_true = entropy_of_one_division(y_real[y_predict]) # left hand side entropy
    s_false, n_false = entropy_of_one_division(y_real[~y_predict]) # right hand side entropy
    s = n_true*1.0/n * s_true + n_false*1.0/n * s_false # overall entropy, again weighted average
    return s

class DecisionTreeClassifier(object):
    def __init__(self, max_depth):
        self.depth = 0
        self.max_depth = max_depth
    
    def fit(self, x, y, par_node={}, depth=0):
        if par_node is None: 
            return None
        elif len(y) == 0:
            return None
        elif self.all_same(y):
            return {'val':y[0]}
        elif depth >= self.max_depth:
            return None
        else: 
            col, cutoff, entropy = self.find_best_split_of_all(x, y)    # find one split given an information gain 
            y_left = y[x[:, col] < cutoff]
            y_right = y[x[:, col] >= cutoff]
            par_node = {'col': iris.feature_names[col], 'index_col':col,
                        'cutoff':cutoff,
                       'val': np.round(np.mean(y))}
            par_node['left'] = self.fit(x[x[:, col] < cutoff], y_left, {}, depth+1)
            par_node['right'] = self.fit(x[x[:, col] >= cutoff], y_right, {}, depth+1)
            self.depth += 1 
            self.trees = par_node
            return par_node
    
    def find_best_split_of_all(self, x, y):
        col = None
        min_entropy = 1
        cutoff = None
        for i, c in enumerate(x.T):
            entropy, cur_cutoff = self.find_best_split(c, y)
            if entropy == 0:    # find the first perfect cutoff. Stop Iterating
                return i, cur_cutoff, entropy
            elif entropy <= min_entropy:
                min_entropy = entropy
                col = i
                cutoff = cur_cutoff
        return col, cutoff, min_entropy
    
    def find_best_split(self, col, y):
        min_entropy = 10
        n = len(y)
        for value in set(col):
            y_predict = col < value
            my_entropy = get_entropy(y_predict, y)
            if my_entropy <= min_entropy:
                min_entropy = my_entropy
                cutoff = value
        return min_entropy, cutoff
    
    def all_same(self, items):
        return all(x == items[0] for x in items)
                                           
    def predict(self, x):
        tree = self.trees
        results = np.array([0]*len(x))
        for i, c in enumerate(x):
            results[i] = self._get_prediction(c)
        return results
    
    def _get_prediction(self, row):
        cur_layer = self.trees
        while cur_layer.get('cutoff'):
            if row[cur_layer['index_col']] < cur_layer['cutoff']:
                cur_layer = cur_layer['left']
            else:
                cur_layer = cur_layer['right']
        else:
            return cur_layer.get('val')

In [ ]:
from sklearn.datasets import load_iris
from pprint import pprint
import matplotlib.pyplot as plt
from sklearn.tree import plot_tree
from sklearn.metrics import accuracy_score

iris = load_iris()

x = iris.data
y = iris.target

clf = DecisionTreeClassifier(max_depth=7)
m = clf.fit(x, y)



pprint(m)

# Visualize the decision tree
plt.figure(figsize=(12, 8))
plot_tree(m, feature_names=iris.feature_names, class_names=iris.target_names, filled=True)
plt.show()

{'col': 'petal width (cm)',
 'cutoff': 1.0,
 'index_col': 3,
 'left': {'val': 0},
 'right': {'col': 'petal width (cm)',
           'cutoff': 1.8,
           'index_col': 3,
           'left': {'col': 'petal length (cm)',
                    'cutoff': 5.0,
                    'index_col': 2,
                    'left': {'col': 'petal width (cm)',
                             'cutoff': 1.7,
                             'index_col': 3,
                             'left': {'val': 1},
                             'right': {'val': 2},
                             'val': 1.0},
                    'right': {'col': 'petal width (cm)',
                              'cutoff': 1.6,
                              'index_col': 3,
                              'left': {'val': 2},
                              'right': {'col': 'sepal length (cm)',
                                        'cutoff': 7.2,
                                        'index_col': 0,
                                        'left

# DECISION TREE (CART) WITH SCRATCH

In [ ]:
import numpy as np
import pandas as pd

# Load the data
data = pd.read_csv("iris.csv")
df = pd.DataFrame(data)

# Print column names
print("Column names in the DataFrame:", df.columns.tolist())

# Function to calculate Gini impurity
def calculate_gini(feature_column):
    # Split the feature into high and low based on a threshold (in this case 4)
    feature_split_high = df[df[feature_column] >= 4]
    feature_split_low = df[df[feature_column] < 4]

    # Get value counts for the species in each split
    counts_high = feature_split_high['Species'].value_counts()
    counts_low = feature_split_low['Species'].value_counts()

    # High split counts for each species
    count_setosa_high = counts_high.get('Iris-setosa', 0)
    count_versicolor_high = counts_high.get('Iris-versicolor', 0)
    count_virginica_high = counts_high.get('Iris-virginica', 0)

    total_high = count_setosa_high + count_versicolor_high + count_virginica_high

    # Calculate Gini for high split
    if total_high > 0:
        gini_setosa_high = count_setosa_high / total_high
        gini_versicolor_high = count_versicolor_high / total_high
        gini_virginica_high = count_virginica_high / total_high
        gini_high = 1 - (gini_setosa_high**2 + gini_versicolor_high**2 + gini_virginica_high**2)
    else:
        gini_high = 0

    # Low split counts for each species
    count_setosa_low = counts_low.get('Iris-setosa', 0)
    count_versicolor_low = counts_low.get('Iris-versicolor', 0)
    count_virginica_low = counts_low.get('Iris-virginica', 0)

    total_low = count_setosa_low + count_versicolor_low + count_virginica_low

    # Calculate Gini for low split
    if total_low > 0:
        gini_setosa_low = count_setosa_low / total_low
        gini_versicolor_low = count_versicolor_low / total_low
        gini_virginica_low = count_virginica_low / total_low
        gini_low = 1 - (gini_setosa_low**2 + gini_versicolor_low**2 + gini_virginica_low**2)
    else:
        gini_low = 0

    # Overall Gini calculation
    total_size = total_high + total_low
    if total_size > 0:
        overall_gini = (total_high / total_size) * gini_high + (total_low / total_size) * gini_low
    else:
        overall_gini = 0

    return gini_high, gini_low, overall_gini

# Select the features excluding 'Species'
features = ['SepalLengthCm', 'SepalWidthCm', 'PetalLengthCm', 'PetalWidthCm']

# Calculate Gini impurity for each feature
for feature in features:
    if feature in df.columns:
        gini_high, gini_low, overall_gini = calculate_gini(feature)
        print(f"Gini impurity for {feature} (>= 4):", gini_high)
        print(f"Gini impurity for {feature} (< 4):", gini_low)
        print(f"Overall Gini impurity for {feature}:", overall_gini)
        print()
    else:
        print(f"Feature '{feature}' not found in DataFrame.")


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Load the data
data = pd.read_csv("iris.csv")
df = pd.DataFrame(data)

# Print column names
print("Column names in the DataFrame:", df.columns.tolist())

# Function to calculate Gini impurity
def calculate_gini(feature_column):
    # Split the feature into high and low based on a threshold (in this case 4)
    feature_split_high = df[df[feature_column] >= 4]
    feature_split_low = df[df[feature_column] < 4]

    # Get value counts for the species in each split
    counts_high = feature_split_high['Species'].value_counts()
    counts_low = feature_split_low['Species'].value_counts()

    # High split counts for each species
    count_setosa_high = counts_high.get('Iris-setosa', 0)
    count_versicolor_high = counts_high.get('Iris-versicolor', 0)
    count_virginica_high = counts_high.get('Iris-virginica', 0)

    total_high = count_setosa_high + count_versicolor_high + count_virginica_high

    # Calculate Gini for high split
    if total_high > 0:
        gini_setosa_high = count_setosa_high / total_high
        gini_versicolor_high = count_versicolor_high / total_high
        gini_virginica_high = count_virginica_high / total_high
        gini_high = 1 - (gini_setosa_high**2 + gini_versicolor_high**2 + gini_virginica_high**2)
    else:
        gini_high = 0

    # Low split counts for each species
    count_setosa_low = counts_low.get('Iris-setosa', 0)
    count_versicolor_low = counts_low.get('Iris-versicolor', 0)
    count_virginica_low = counts_low.get('Iris-virginica', 0)

    total_low = count_setosa_low + count_versicolor_low + count_virginica_low

    # Calculate Gini for low split
    if total_low > 0:
        gini_setosa_low = count_setosa_low / total_low
        gini_versicolor_low = count_versicolor_low / total_low
        gini_virginica_low = count_virginica_low / total_low
        gini_low = 1 - (gini_setosa_low**2 + gini_versicolor_low**2 + gini_virginica_low**2)
    else:
        gini_low = 0

    # Overall Gini calculation
    total_size = total_high + total_low
    if total_size > 0:
        overall_gini = (total_high / total_size) * gini_high + (total_low / total_size) * gini_low
    else:
        overall_gini = 0

    return gini_high, gini_low, overall_gini

# Select the features excluding 'Species'
features = ['SepalLengthCm', 'SepalWidthCm', 'PetalLengthCm', 'PetalWidthCm']

# Lists to store the results for visualization
gini_high_list = []
gini_low_list = []
overall_gini_list = []
feature_names = []

# Calculate Gini impurity for each feature
for feature in features:
    if feature in df.columns:
        gini_high, gini_low, overall_gini = calculate_gini(feature)
        print(f"Gini impurity for {feature} (>= 4):", gini_high)
        print(f"Gini impurity for {feature} (< 4):", gini_low)
        print(f"Overall Gini impurity for {feature}:", overall_gini)
        print()

        # Store results for visualization
        gini_high_list.append(gini_high)
        gini_low_list.append(gini_low)
        overall_gini_list.append(overall_gini)
        feature_names.append(feature)
    else:
        print(f"Feature '{feature}' not found in DataFrame.")

# Visualization using bar plot
fig, ax = plt.subplots(figsize=(10, 6))

# Set the width of the bars
bar_width = 0.25
index = np.arange(len(feature_names))

# Plot the Gini values
bar1 = ax.bar(index, gini_high_list, bar_width, label='Gini >= 4', color='lightcoral')
bar2 = ax.bar(index + bar_width, gini_low_list, bar_width, label='Gini < 4', color='skyblue')
bar3 = ax.bar(index + 2 * bar_width, overall_gini_list, bar_width, label='Overall Gini', color='mediumseagreen')

# Add labels and title
ax.set_xlabel('Features', fontsize=14)
ax.set_ylabel('Gini Impurity', fontsize=14)
ax.set_title('Gini Impurity for Iris Dataset Features', fontsize=16)
ax.set_xticks(index + bar_width)
ax.set_xticklabels(feature_names, fontsize=12)
ax.legend(fontsize=12)

# Display the Gini values above the bars
def add_labels(bars):
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2, height + 0.02, f'{height:.2f}', ha='center', va='bottom', fontsize=10)

# Call the function to display the Gini values
add_labels(bar1)
add_labels(bar2)
add_labels(bar3)

# Display the plot
plt.tight_layout()
plt.show()


# RANDOM FOREST WITH SCRATCH

In [ ]:
# RANDOM FOREST

import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

In [ ]:
iris = pd.read_csv("iris.csv") #Load Data
iris.drop('Id',inplace=True,axis=1) #Drop Id column

In [ ]:
iris.head().style.background_gradient(cmap =sns.light_palette("seagreen", as_cmap=True)
)

In [ ]:
import numpy as np
import pandas as pd
iris_df = pd.read_csv('iris.csv')
print(iris_df.head())


In [ ]:
X = iris_df.drop(columns=['Id', 'Species']).values 
y = iris_df['Species'].factorize()[0]  

print(f"Features shape: {X.shape}")
print(f"Labels shape: {y.shape}")


In [ ]:
def train_test_split_custom(X, y, test_size=0.3, random_state=None):
    if random_state is not None:
        np.random.seed(random_state) 
    
    n_samples = X.shape[0]
    indices = np.random.permutation(n_samples)  
    split_index = int(n_samples * (1 - test_size))  
    train_indices = indices[:split_index]
    test_indices = indices[split_index:]
    
    X_train = X[train_indices]
    y_train = y[train_indices]
    X_test = X[test_indices]
    y_test = y[test_indices]
    
    return X_train, X_test, y_train, y_test

X_train, X_test, y_train, y_test = train_test_split_custom(X, y, test_size=0.3, random_state=42)

print(f"Training Features shape: {X_train.shape}")
print(f"Training Labels shape: {y_train.shape}")
print(f"Testing Features shape: {X_test.shape}")
print(f"Testing Labels shape: {y_test.shape}")


In [ ]:
def gini_impurity(y):
    unique_classes, counts = np.unique(y, return_counts=True)
    probabilities = counts / len(y)
    return 1 - np.sum(probabilities ** 2)

def split_dataset(X, y, feature_index, threshold):
    left_indices = X[:, feature_index] <= threshold
    right_indices = X[:, feature_index] > threshold
    
    X_left, y_left = X[left_indices], y[left_indices]
    X_right, y_right = X[right_indices], y[right_indices]
    
    return X_left, X_right, y_left, y_right


In [ ]:
def best_split(X, y):
    best_gini = float('inf')
    best_feature = None
    best_threshold = None
    n_samples, n_features = X.shape

    for feature_index in range(n_features):
        thresholds = np.unique(X[:, feature_index])
        for threshold in thresholds:
            X_left, X_right, y_left, y_right = split_dataset(X, y, feature_index, threshold)
            if len(y_left) == 0 or len(y_right) == 0:
                continue
                
            gini_left = gini_impurity(y_left)
            gini_right = gini_impurity(y_right)
            weighted_gini = (len(y_left) / n_samples) * gini_left + (len(y_right) / n_samples) * gini_right
            
            if weighted_gini < best_gini:
                best_gini = weighted_gini
                best_feature = feature_index
                best_threshold = threshold
    
    return best_feature, best_threshold

class DecisionTreeNode:
    def __init__(self, feature=None, threshold=None, left=None, right=None, value=None):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value

def build_tree(X, y, max_depth=10, depth=0):
    n_samples, n_features = X.shape
    unique_classes, class_counts = np.unique(y, return_counts=True)
    
    if len(unique_classes) == 1 or depth == max_depth:
        leaf_value = unique_classes[np.argmax(class_counts)]
        return DecisionTreeNode(value=leaf_value)

    feature_index, threshold = best_split(X, y)

    if feature_index is None:
        leaf_value = unique_classes[np.argmax(class_counts)]
        return DecisionTreeNode(value=leaf_value)

    X_left, X_right, y_left, y_right = split_dataset(X, y, feature_index, threshold)
    left_child = build_tree(X_left, y_left, max_depth, depth + 1)
    right_child = build_tree(X_right, y_right, max_depth, depth + 1)
    
    return DecisionTreeNode(feature=feature_index, threshold=threshold, left=left_child, right=right_child)


In [ ]:
def predict_tree(node, X):
    if node.value is not None:
        return node.value
    
    if X[node.feature] <= node.threshold:
        return predict_tree(node.left, X)
    else:
        return predict_tree(node.right, X)


In [ ]:
def bootstrap_sample(X, y):
    n_samples = X.shape[0]
    indices = np.random.choice(n_samples, size=n_samples, replace=True)
    return X[indices], y[indices]

class RandomForest:
    def __init__(self, n_trees=10, max_depth=10):
        self.n_trees = n_trees
        self.max_depth = max_depth
        self.trees = []
    
    def fit(self, X, y):
        self.trees = []
        for _ in range(self.n_trees):
            X_sample, y_sample = bootstrap_sample(X, y)
            tree = build_tree(X_sample, y_sample, max_depth=self.max_depth)
            self.trees.append(tree)
    
    def predict(self, X):
        tree_predictions = np.array([predict_tree(tree, sample) for tree in self.trees for sample in X])
        tree_predictions = tree_predictions.reshape(self.n_trees, len(X))
        
        final_predictions = []
        for i in range(len(X)):
            votes = {}
            for tree_prediction in tree_predictions[:, i]:
                if tree_prediction not in votes:
                    votes[tree_prediction] = 1
                else:
                    votes[tree_prediction] += 1
            final_predictions.append(max(votes, key=votes.get))
        
        return np.array(final_predictions)


In [ ]:
rf = RandomForest(n_trees=10, max_depth=10)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)

accuracy = np.mean(y_pred == y_test)
print(f"Accuracy: {accuracy}")


In [ ]:
def calculate_feature_importance(trees, n_features):
    feature_importance = np.zeros(n_features)
    
    def count_splits(node):
        if node.feature is not None:
            feature_importance[node.feature] += 1
            count_splits(node.left)
            count_splits(node.right)
    
    for tree in trees:
        count_splits(tree)
    
    feature_importance /= np.sum(feature_importance)
    
    return feature_importance

n_features = X_train.shape[1]
feature_importance = calculate_feature_importance(rf.trees, n_features)

features = iris_df.columns[1:-1]  
plt.figure(figsize=(8, 6))
plt.barh(features, feature_importance, color='purple')
plt.xlabel('Feature Importance')
plt.title('Feature Importance in Random Forest')
plt.show()


In [ ]:
# NAIVE BAYES

from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np
from collections import Counter
import math
from scipy.stats import norm



In [ ]:

iris = pd.read_csv('iris.csv')



In [ ]:
iris.head()

In [ ]:
iris.describe()

In [ ]:
iris.info()

In [ ]:
target_category = iris["Species"].unique()
target_category=list(map(str,target_category))
print(target_category)

In [ ]:
iris.groupby("Species").Species.count().plot.bar(ylim=0)

In [ ]:
species = iris.Species
data = iris.drop(columns=['Species','Id']) 

In [ ]:
data.head()

In [ ]:
iris['Category'] = iris['Species'].factorize()[0]
category = iris['Category']
iris.head()

In [ ]:
#split dataset into test set(20%) and train set(80%) using stratify to split into equal size
data_train,data_test,species_train,species_test = train_test_split(data,category, test_size = 0.2, stratify = category,random_state=1)
print(np.bincount(species_train))

In [ ]:
newIris = pd.DataFrame(np.column_stack([data_train,species_train]))

In [ ]:
setosa = newIris[newIris[4] == 0]
versicolor = newIris[newIris[4]==1]       
virginica  = newIris[newIris[4]==2] 
newIris = pd.concat([setosa,versicolor,virginica])

In [ ]:
setosa_data=newIris[0:40] 
versicolor_data=newIris[40:80]
virginica_data=newIris[80:120]

In [ ]:
setosa_mean = setosa_data.mean()
versicolor_mean = versicolor_data.mean()
virginica_mean= virginica_data.mean()

In [ ]:
setosa_std = setosa_data.std()
versicolor_std = versicolor_data.std()
virginica_std = virginica_data.std()

In [ ]:
x =[]
likelihood = []
   
for j in range(len(newIris)):
    distribution = 1
    if(j<40):
        mean=setosa_mean
        std = setosa_std
    if(j>=40 and j<80):
        mean=versicolor_mean
        std = versicolor_std
    if(j>=80 and j<120):
        mean=virginica_mean
        std = virginica_std    
    
    for i in range(4):
        x = newIris.iloc[j] 
        a= ((x[i]- mean[i])**2)/(2*std[i]**2)
        b= math.sqrt(2*math.pi*(std[i]**2))
        y = math.exp(-a)/b
        distribution= distribution*y  
    likelihood.append(distribution)
    x=[]  
print(likelihood)

In [ ]:
setosa_priori = len(setosa_data)/len(newIris)
versicolor_priori = len(versicolor_data)/len(newIris)
virginica_priori = len(virginica_data)/len(newIris)

print(setosa_priori)
print(versicolor_priori)
print(virginica_priori)

In [ ]:
newTest = pd.DataFrame(np.column_stack([data_test,species_test]))

In [ ]:
setosa = newTest[newTest[4] == 0]
versicolor = newTest[newTest[4]==1]       
virginica  = newTest[newTest[4]==2] 
newTest = pd.concat([setosa,versicolor,virginica])

In [ ]:
testLikelihood =[]
x=[]
testPosterior=[]
posteriorSpecies =[]


for j in range(len(newTest)):
    for c in range(3): 
        if (c==0):
            mean = setosa_mean
            std = setosa_std
            priori = setosa_priori
        if (c == 1):
            mean= versicolor_mean
            std = versicolor_std
            priori = versicolor_priori
        if(c == 2):
            mean= virginica_mean
            std = virginica_std
            priori = virginica_priori
        distribution = 1       
        for i in range(4):
            x = newTest.iloc[j] 
            a= ((x[i]- mean[i])**2)/(2*std[i]**2)
            b= math.sqrt(2*math.pi*(std[i]**2))
            y = math.exp(-a)/b
            distribution= distribution*y
        x=[]    
        testLikelihood.append(distribution)
        posterior = testLikelihood[c]* priori    #Calculate poterior values
        testPosterior.append(posterior)
        maxPosterior = testPosterior.index(max(testPosterior))   #finds the maximum value
    posteriorSpecies.append(maxPosterior)
   

    testLikelihood =[]    
    testPosterior=[]

In [ ]:
print("Species of original test data") 
species_test = list(map(int,newTest[4]))
print(species_test)

print("Species of Maximum Posterior values")    
print(posteriorSpecies)

In [ ]:
similar=0
for i in range(len(posteriorSpecies)):
    if(species_test[i] == posteriorSpecies[i]):
        similar +=1
accuracy = similar/(i+1)*100

print(accuracy)

In [ ]:
index = np.arange(len(species_test))

# Plotting the original vs predicted species
plt.figure(figsize=(10, 6))

# Plot original test species
plt.scatter(index, species_test, color='blue', label='Original Species')

# Plot predicted species from Maximum Posterior values
plt.scatter(index, posteriorSpecies, color='red', marker='x', label='Predicted Species')

# Adding labels and title
plt.xlabel('Test Data Index')
plt.ylabel('Species')
plt.title('Comparison of Original and Predicted Species')
plt.legend()

# Show the plot
plt.show()